# Recommendation System Evaluation

This notebook demonstrates comprehensive evaluation and visualization of Content-Based, Collaborative Filtering, and Hybrid recommendation models.

In [1]:
%cd ..

/home/jovyan/project/film-recommendation


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_processing.data_loader import MovieLensDataLoader
from src.models.content_based import ContentBasedRecommender, ContentBasedConfig
from src.models.collaborative_filtering import CollaborativeFiltering
from src.models.hybrid import HybridRecommender
from src.models.cascading_hybrid import CascadingHybridRecommender
from src.models.popular_baseline_model import PopularityBaseline
from src.evaluation.evaluator import RecommendationEvaluator
from src.evaluation.visualisation import RecommendationVisualizer
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

In [3]:
## 1. Load and Prepare Data

In [4]:
loader = MovieLensDataLoader("ml-latest")
data_dict = loader.load_data()
await loader.letterboxd_data_async(max_concurrent_requests=100)

movies_df = pd.DataFrame(loader.movie_data)
ratings_df = data_dict['ratings']
genre_features = loader.preprocess_movies()
movies_df = pd.concat([movies_df, genre_features], axis=1)
movies_df = movies_df.dropna().reset_index(drop=True)

print(f"Movies: {len(movies_df)}")
print(f"Ratings: {len(ratings_df)}")
print(f"Users: {ratings_df['userId'].nunique()}")

INFO:src.data_processing.data_loader:Loading MovieLens dataset...
INFO:src.data_processing.data_loader:Loading existing data from cache: data/processed/movie_metadata.parquet
INFO:src.data_processing.data_loader:Found 1180 missing movies with valid TMDB IDs. Fetching...
INFO:src.data_processing.data_loader:DOWNLOAD SUMMARY: Requested: 1180 | Saved: 0 | Failed: 1180


Movies: 82033
Ratings: 33832162
Users: 330975


## 2. Split Data (Temporal Split)

In [5]:
ratings_df = ratings_df.sort_values('timestamp')
split_idx = int(len(ratings_df) * 0.8)

train_df = ratings_df.iloc[:split_idx].copy()
test_df = ratings_df.iloc[split_idx:].copy()

print(f"Train set: {len(train_df)} ratings")
print(f"Test set: {len(test_df)} ratings")
print(f"Train users: {train_df['userId'].nunique()}")
print(f"Test users: {test_df['userId'].nunique()}")

Train set: 27065729 ratings
Test set: 6766433 ratings
Train users: 280422
Test users: 58661


## 3. Train Models

In [6]:
#tags_df = pd.read_csv('/home/aquathirsty/Desktop/PFE/film-recommendations/data/raw/movielens/ml-latest-small/tags.csv')

config = ContentBasedConfig(
    main_actor_weight=0.3,
    director_weight=0.3,
    cast_weight=0.3,
    keywords_weight=0.6,
    numerical_weight=0.1,
    similarity_threshold=0.15,
    top_k_default=20
)

cb_model = ContentBasedRecommender(config=config)
cb_model.fit(movies_df=movies_df, ratings_df=train_df)
print("Content-Based model with tags trained successfully!")

2026-06-28 15:37:38.486 | INFO     | src.models.content_based:fit:697 - Starting model fitting process
--- Step: Validation ---
  Wall clock time : 0.00 s
  CPU time (user) : 0.00 s
  Memory RSS      : 5708.6 MB
  Memory VMS      : 13126.8 MB
  CPU usage       : 0.0 %
--- Step: Preprocessing ---
  Wall clock time : 0.17 s
  CPU time (user) : 0.17 s
  Memory RSS      : 5719.7 MB
  Memory VMS      : 13136.8 MB
  RSS change      : +11.1 MB
  CPU usage       : 0.0 %
--- Step: Feature Matrix ---
  Wall clock time : 4.18 s
  CPU time (user) : 4.26 s
  Memory RSS      : 5799.3 MB
  Memory VMS      : 13462.8 MB
  RSS change      : +79.6 MB
  CPU usage       : 0.0 %
Similarity batches: 100%|██████████| 55/55 [01:50<00:00,  2.01s/batch]
--- Step: Similarity Matrix ---
  Wall clock time : 115.30 s
  CPU time (user) : 115.02 s
  Memory RSS      : 8170.6 MB
  Memory VMS      : 15834.0 MB
  RSS change      : +2371.4 MB
  CPU usage       : 0.0 %
--- Step: Index Mappings ---
  Wall clock time : 115.30

Content-Based model with tags trained successfully!


In [7]:
cf_model = CollaborativeFiltering(k_components=110, random_state=42)
cf_model.fit(df_ratings=train_df)
print("Collaborative Filtering model trained successfully!")

--- Step: Filtering & Mapping ---
  Wall clock time : 0.32 s
  CPU time (user) : 0.32 s
  Memory RSS      : 10232.7 MB
  Memory VMS      : 17906.5 MB
  CPU usage       : 0.0 %
  n_users         : 280422
  n_items         : 49383
  n_ratings       : 27065729
  n_popular_items : 18913
--- Step: Data Preparation ---
  Wall clock time : 2.74 s
  CPU time (user) : 2.74 s
  Memory RSS      : 10595.9 MB
  Memory VMS      : 18269.7 MB
  RSS change      : +363.3 MB
  CPU usage       : 0.0 %
  global_mean     : 3.5320537090301514
  n_factors       : 50
  sparsity        : 0.1954%
--- Step: SGD Optimization ---
  Wall clock time : 242.39 s
  CPU time (user) : 241.54 s
  Memory RSS      : 10630.2 MB
  Memory VMS      : 18270.1 MB
  RSS change      : +34.2 MB
  CPU usage       : 0.0 %
  n_epochs        : 20
  lr              : 0.004999999888241291
  reg             : 0.019999999552965164
--- Step: Finalization ---
  Wall clock time : 242.40 s
  CPU time (user) : 241.55 s
  Memory RSS      : 10630.2

Collaborative Filtering model trained successfully!


In [8]:
hybrid_model = HybridRecommender(
    cf_model=cf_model,
    cb_model=cb_model,
    alpha=0.8
)
hybrid_model.fitted(cf_model=cf_model, cb_model=cb_model, movies_df=movies_df, ratings_df=train_df)
print("Hybrid model trained successfully!")

Hybrid model trained successfully!


In [9]:
pop_model = PopularityBaseline()
pop_model.fit(train_df)
print("Popularity model trained successfully!")

Popularity model trained successfully!


In [10]:
cascading_hybrid_model = CascadingHybridRecommender(
    primary_model=cf_model,
    secondary_model=cb_model,
    primary_k=50
)
cascading_hybrid_model.fitted(primary_model=cf_model, secondary_model=cb_model)
print("Cascading hybrid model trained successfully!")

Cascading hybrid model trained successfully!


## 4. Evaluate Models

In [ ]:
models = {
    'Content-Based': cb_model,
    'Collaborative': cf_model,
    'Hybrid': hybrid_model,
    'Cascading Hybrid': cascading_hybrid_model,
    'Popularity': pop_model,
}

evaluator = RecommendationEvaluator(
    models=models,
    train_df=train_df,
    test_df=test_df,
    relevance_threshold=4.0,
    user_sample_size=None,
    random_state=42
)

results_df = evaluator.evaluate_all_models(
    k_values=[5, 10, 20],
    max_recommendations=20
)

print("Evaluation completed!")
print(f"Results shape: {results_df.shape}")

2026-06-28 15:44:05.088 | INFO     | src.evaluation.evaluator:evaluate_model:158 - Evaluating 'Content-Based'
2026-06-28 15:44:52.113 | INFO     | src.evaluation.evaluator:evaluate_model:230 - 'Content-Based' done in 47.0s
2026-06-28 15:44:52.114 | INFO     | src.evaluation.evaluator:evaluate_model:158 - Evaluating 'Collaborative'
Collaborative:  83%|████████▎ | 24/29 [01:48<00:17,  3.42s/it]

In [ ]:
results_df

## 5. Visualizations

In [ ]:
visualizer = RecommendationVisualizer(results_df)

### 5.1 Precision@K Trend

In [ ]:
fig = visualizer.plot_metric_trend('precision', figsize=(10, 6))
plt.show()

### 5.2 Recall@K Trend

In [ ]:
fig = visualizer.plot_metric_trend('recall', figsize=(10, 6))
plt.show()

### 5.3 NDCG@K Trend

In [ ]:
fig = visualizer.plot_metric_trend('ndcg', figsize=(10, 6))
plt.show()

### 5.4 Model Comparison at K=10

In [ ]:
fig = visualizer.plot_model_comparison(k=10, figsize=(12, 7))
plt.show()

### 5.5 Performance Heatmap at K=10

In [ ]:
fig = visualizer.plot_all_metrics_heatmap(k=10, figsize=(10, 8))
plt.show()

### 5.6 Coverage vs Novelty Trade-off

In [ ]:
fig = visualizer.plot_coverage_novelty_tradeoff(k=10, figsize=(10, 7))
plt.show()

### 5.7 Radar Chart Comparison

In [ ]:
fig = visualizer.plot_radar_chart(k=10, figsize=(10, 10))
plt.show()

## 6. Save All Plots

In [ ]:
visualizer.save_all_plots(output_dir="evaluation_plots")
print("All plots saved to evaluation_plots/")

## 7. Summary Statistics

In [ ]:
print("=" * 80)
print("EVALUATION SUMMARY (K=10)")
print("=" * 80)

k10_results = results_df[results_df['k'] == 10]

for model in k10_results['model'].unique():
    model_data = k10_results[k10_results['model'] == model].iloc[0]
    print(f"\n{model.upper()}:")
    print(f"  Precision@10: {model_data['precision']:.4f}")
    print(f"  Recall@10:    {model_data['recall']:.4f}")
    print(f"  NDCG@10:      {model_data['ndcg']:.4f}")
    print(f"  MAP@10:       {model_data['map']:.4f}")
    print(f"  MRR:          {model_data['mrr']:.4f}")
    print(f"  Novelty:      {model_data['novelty']:.4f}")
    print(f"  Coverage:     {model_data['coverage']:.4f}")

print("\n" + "=" * 80)